# Re-extract face embeddings with `enet_b0_8_va_mtl`

This notebook re-runs the face-embedding extraction pipeline from `face_encoding_additional_encoders.ipynb` but with the **`enet_b0_8_va_mtl`** EmotiEffNet backbone — the same model used by the upstream ABAW8 EMI training notebook (`EmotiEffLib/training_and_examples/ABAW/ABAW8/emi.ipynb`) that achieves Pearson ≈ 0.1776 on the EMI track.

**The new embeddings overwrite the existing directory** so that `abaw8_emi_face_model_reproduction.ipynb` (and any other notebook pointing at that directory) can be re-run without changing its `FACE_DIR`. The old `enet_b0_8_best_afew` files in that directory will be replaced.

All other settings (MTCNN detector, fps=5 sampling, margin, batch sizes, NPZ schema) are kept identical to `face_encoding_additional_encoders.ipynb`.

In [ ]:
from pathlib import Path

# ========== Paths ==========
DATA_DIR = Path("/home/danila/networks/data")

FRAMES_DIR = DATA_DIR / "frames-1"
TRAIN_CSV  = DATA_DIR / "train_split.csv"
VALID_CSV  = DATA_DIR / "valid_split.csv"
ID_WIDTH   = 5

# Backbone used by Savchenko CVPRW 2025 / EmotiEffLib upstream EMI notebook.
EMOTIEFF_MODEL = "enet_b0_8_va_mtl"

# IMPORTANT: we deliberately reuse the SAME output directory as the previous
# `enet_b0_8_best_afew` extraction so that downstream notebooks (e.g.
# `abaw8_emi_face_model_reproduction.ipynb`) keep working without any path
# changes. The previously extracted .npz files will be OVERWRITTEN.
OUT_DIR = DATA_DIR / "embeddings" / "faces_emotiefflib_enet_b0_8_best_afew_fps5_v1"
OUT_DIR.mkdir(parents=True, exist_ok=True)
INDEX_PATH = OUT_DIR / "index.parquet"

# ========== FPS ==========
ORIG_FPS   = 15
TARGET_FPS = 5
assert ORIG_FPS % TARGET_FPS == 0, "invalid stride"
FRAME_STRIDE = ORIG_FPS // TARGET_FPS

# ========== Face detection (MTCNN) ==========
DETECTOR_DEVICE = "cuda"
MTCNN_IMAGE_BATCH = 16
FACE_MIN_PROB = 0.90
MTCNN_MIN_FACE_SIZE = 40
MTCNN_THRESHOLDS = (0.6, 0.7, 0.7)
FACE_MARGIN = 0.20

# ========== EmotiEffLib encoder ==========
EMOTIEFF_ENGINE = "torch"
ENCODER_DEVICE  = "cuda"

# ========== Encoding batching ==========
FACE_ENC_BATCH = 64
EMB_SAVE_DTYPE = "float16"

# ========== Output options ==========
SAVE_LOGITS = True
# Force re-extraction: we WANT to overwrite the previous backbone's files.
SKIP_IF_EXISTS = False

In [ ]:
import os, json, math
import numpy as np
import pandas as pd
import cv2
import torch

from PIL import Image
from tqdm.auto import tqdm

from facenet_pytorch import MTCNN
from emotiefflib.facial_analysis import EmotiEffLibRecognizer, get_model_list

print("CUDA available:", torch.cuda.is_available())
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
assert FRAMES_DIR.exists(), f"FRAMES_DIR not found: {FRAMES_DIR}"

mtcnn = MTCNN(
    keep_all=True,
    device=DEVICE,
    thresholds=MTCNN_THRESHOLDS,
    min_face_size=MTCNN_MIN_FACE_SIZE,
    post_process=False,
)

assert EMOTIEFF_MODEL in set(get_model_list()), (
    f"Unknown model {EMOTIEFF_MODEL}. Available: {get_model_list()}"
)
encoder = EmotiEffLibRecognizer(
    engine=EMOTIEFF_ENGINE,
    model_name=EMOTIEFF_MODEL,
    device=DEVICE if EMOTIEFF_ENGINE == "torch" else "cpu",
)

dummy = np.zeros((224, 224, 3), dtype=np.uint8)
D = int(encoder.extract_features(dummy).shape[1])
print("Embedding dim D =", D, "| model =", EMOTIEFF_MODEL, "| out_dir =", OUT_DIR)

In [ ]:
from typing import List, Tuple, Optional, Dict, Any

def read_rgb(path: Path) -> np.ndarray:
    bgr = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if bgr is None:
        raise FileNotFoundError(f"Failed to read: {path}")
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

def parse_frame_number(path: Path) -> int:
    return int(path.stem)

def list_frames(video_id: str) -> List[Path]:
    d = FRAMES_DIR / video_id
    if not d.exists():
        return []
    return sorted(d.glob("*.jpg"))

def sample_frames_5fps(frames: List[Path]) -> List[Path]:
    if not frames:
        return []
    return frames[::FRAME_STRIDE]

def largest_box(boxes: np.ndarray) -> int:
    areas = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
    return int(np.argmax(areas))

def crop_face_with_margin(img_rgb: np.ndarray, box_xyxy: np.ndarray, margin: float) -> np.ndarray:
    h, w = img_rgb.shape[:2]
    x1, y1, x2, y2 = box_xyxy.astype(np.float32)
    bw = x2 - x1; bh = y2 - y1
    x1 -= margin * bw; y1 -= margin * bh
    x2 += margin * bw; y2 += margin * bh
    x1 = int(max(0, math.floor(x1)))
    y1 = int(max(0, math.floor(y1)))
    x2 = int(min(w, math.ceil(x2)))
    y2 = int(min(h, math.ceil(y2)))
    if x2 <= x1 or y2 <= y1:
        raise ValueError("Invalid crop box after margin")
    return img_rgb[y1:y2, x1:x2, :]

def chunked(seq, n: int):
    for i in range(0, len(seq), n):
        yield seq[i:i+n]

In [ ]:
def process_one_video(video_id: str) -> Optional[Dict[str, Any]]:
    out_npz = OUT_DIR / f"{video_id}.npz"
    out_meta = OUT_DIR / f"{video_id}.json"

    if SKIP_IF_EXISTS and out_npz.exists():
        try:
            meta = json.loads(out_meta.read_text())
            meta["video_id"] = video_id
            meta["npz_path"] = str(out_npz)
            return meta
        except Exception:
            return {"video_id": video_id, "npz_path": str(out_npz), "skipped": True}

    frames = list_frames(video_id)
    if not frames:
        return None

    frames = sample_frames_5fps(frames)
    if not frames:
        return None

    T = len(frames)
    frame_numbers = np.array([parse_frame_number(p) for p in frames], dtype=np.int32)
    timestamps_sec = (frame_numbers - 1).astype(np.float32) / float(ORIG_FPS)

    embeddings = np.zeros((T, D), dtype=np.float16)
    bbox_xyxy  = np.full((T, 4), -1.0, dtype=np.float32)
    face_prob  = np.zeros((T,), dtype=np.float32)
    face_found = np.zeros((T,), dtype=bool)

    logits = None
    K = None
    W = getattr(encoder, "classifier_weights", None)
    b = getattr(encoder, "classifier_bias", None)

    with torch.inference_mode():
        for batch_idx, batch_paths in enumerate(chunked(frames, MTCNN_IMAGE_BATCH)):
            batch_rgb = [read_rgb(p) for p in batch_paths]
            batch_pil = [Image.fromarray(im) for im in batch_rgb]

            boxes_list, probs_list = mtcnn.detect(batch_pil)

            face_imgs = []
            face_map  = []

            for j, (img_rgb, boxes, probs) in enumerate(zip(batch_rgb, boxes_list, probs_list)):
                global_i = batch_idx * MTCNN_IMAGE_BATCH + j
                if boxes is None or probs is None:
                    continue
                keep = probs >= FACE_MIN_PROB
                if not np.any(keep):
                    continue
                boxes_k = boxes[keep]; probs_k = probs[keep]
                k = largest_box(boxes_k)
                box = boxes_k[k]; pr = float(probs_k[k])
                try:
                    crop = crop_face_with_margin(img_rgb, box, FACE_MARGIN)
                except Exception:
                    continue
                face_imgs.append(crop)
                face_map.append((global_i, box.astype(np.float32), pr))

            if face_imgs:
                feats_all = []
                for faces_chunk in chunked(face_imgs, FACE_ENC_BATCH):
                    feats = encoder.extract_features(faces_chunk)
                    feats_all.append(feats)
                feats_all = np.concatenate(feats_all, axis=0)

                for (global_i, box, pr), feat in zip(face_map, feats_all):
                    face_found[global_i] = True
                    face_prob[global_i]  = pr
                    bbox_xyxy[global_i]  = box
                    embeddings[global_i] = feat.astype(np.float16)

                if SAVE_LOGITS and (W is not None) and (b is not None):
                    if logits is None:
                        K = int(W.shape[0])
                        logits = np.zeros((T, K), dtype=np.float16)
                    f32 = feats_all.astype(np.float32)
                    l32 = (f32 @ W.T.astype(np.float32)) + b.astype(np.float32)
                    for (global_i, _, _), l in zip(face_map, l32):
                        logits[global_i] = l.astype(np.float16)

    meta = {
        "video_id": video_id,
        "npz_path": str(out_npz),
        "model": EMOTIEFF_MODEL,
        "engine": EMOTIEFF_ENGINE,
        "orig_fps": ORIG_FPS,
        "target_fps": TARGET_FPS,
        "frame_stride": FRAME_STRIDE,
        "embedding_dim": int(D),
        "frames_total": int(T),
        "faces_found": int(face_found.sum()),
        "save_logits": bool(SAVE_LOGITS and logits is not None),
        "logits_dim": int(K) if logits is not None else None,
    }

    save_dict = dict(
        timestamps_sec=timestamps_sec,
        frame_numbers=frame_numbers,
        bbox_xyxy=bbox_xyxy,
        face_prob=face_prob,
        face_found=face_found,
        embeddings=embeddings,
    )
    if logits is not None:
        save_dict["emotion_logits"] = logits

    np.savez_compressed(out_npz, **save_dict)
    out_meta.write_text(json.dumps(meta, ensure_ascii=False, indent=2))
    return meta

In [ ]:
train_df = pd.read_csv(TRAIN_CSV, dtype={"Filename": str})
valid_df = pd.read_csv(VALID_CSV, dtype={"Filename": str})
train_df["Filename"] = train_df["Filename"].str.zfill(ID_WIDTH)
valid_df["Filename"] = valid_df["Filename"].str.zfill(ID_WIDTH)

all_ids = pd.concat([train_df[["Filename"]], valid_df[["Filename"]]], axis=0)["Filename"].astype(str).unique().tolist()
all_ids = sorted(all_ids)

print("Total videos:", len(all_ids))
print("Output dir:", OUT_DIR, "(will be overwritten with", EMOTIEFF_MODEL, "features)")

metas, failed = [], []
for vid in tqdm(all_ids, desc=f"Extract {EMOTIEFF_MODEL}"):
    try:
        m = process_one_video(vid)
        if m is None:
            failed.append({"video_id": vid, "reason": "no_frames"})
        else:
            metas.append(m)
    except Exception as e:
        failed.append({"video_id": vid, "reason": repr(e)})

meta_df = pd.DataFrame(metas)
fail_df = pd.DataFrame(failed)
print("OK:", len(meta_df), "Failed:", len(fail_df))
meta_df.head()

In [ ]:
meta_df.to_parquet(INDEX_PATH, index=False)
print("Saved index:", INDEX_PATH)

if len(fail_df):
    fail_path = OUT_DIR / "failed.csv"
    fail_df.to_csv(fail_path, index=False)
    print("Saved failures:", fail_path)

In [ ]:
test_id = meta_df.iloc[0]["video_id"]
npz_path = OUT_DIR / f"{test_id}.npz"
data = np.load(npz_path, allow_pickle=False)
print("keys:", data.files)
print("timestamps:", data["timestamps_sec"].shape, data["timestamps_sec"][:5])
print("embeddings:", data["embeddings"].shape, data["embeddings"].dtype)
print("faces found:", data["face_found"].sum(), "/", data["face_found"].shape[0])